<a href="https://colab.research.google.com/github/San-Maansson/Stock_Analysis/blob/main/max_diversification_port.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
"""
Maximum Diversification Portfolio (MDP)
"""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
import yfinance as yf

In [3]:
#DJIA constituents as of 2025-05-29
TICKERS = [
    "MMM",  # 3M
    "AXP",  # American Express
    "AMGN", # Amgen
    "AMZN", # Amazon
    "AAPL", # Apple
    "BA",   # Boeing
    "CAT",  # Caterpillar
    "CVX",  # Chevron
    "CSCO", # Cisco
    "KO",   # Coca-Cola
    "DIS",  # Disney
    "GS",   # Goldman Sachs
    "HD",   # Home Depot
    "HON",  # Honeywell
    "IBM",  # IBM
    "JNJ",  # Johnson & Johnson
    "JPM",  # JPMorgan Chase
    "MCD",  # McDonald's
    "MRK",  # Merck
    "MSFT", # Microsoft
    "NKE",  # Nike
    "NVDA", # Nvidia
    "PG",   # Procter & Gamble
    "CRM",  # Salesforce
    "SHW",  # Sherwin-Williams
    "TRV",  # Travelers
    "UNH",  # United Health Group
    "VZ",   # Verizon
    "V",    # Visa
    "WMT",  # Walmart
]

In [4]:
def data_download(TICKERS: list, period="1y", interval="1d"):
    """
    Parameters
    ----------
    TICKERS: list, tickers to download
    period: str, default one year "1y"
    interval: str, default one day "1d"

    Returns
    -------
    df_ret: DataFrame, daily log returns of stocks
    """
    #The group_by="ticker" returns a column hierarchy per ticker
    data = yf.download(
        tickers=TICKERS,
        period=period,
        interval=interval,
        auto_adjust=True,
        progress=False,
        group_by="ticker",
    )

    #Extract adjusted close
    df = pd.DataFrame({t: data[t]["Close"] if "Close" in data[t] else data[t]["Adj Close"]
         for t in TICKERS if t in data.columns.get_level_values(0)})

    #Compute daily log returns
    df_ret = np.log(df).diff().dropna()

    #Save data
    df_ret.to_csv("downloaded_data.csv", index=False)
    return df_ret


def moments(df, TRAD_DAYS=252):
    """
    Parameters
    ----------
    df:         Dataframe, log returns of n assets
    TRAD_DAYS:  int, number of trading days, default 252
    Returns
    -------
    assets:     list, names of securities
    n:          int, number of assets
    mu:         float, expected returns
    Sigma:      float, covariance matrix
    TRAD_DAYS:  int, number of trading days (for later calculations)
    """
    #Assumes daily data
    mu_annual = df.mean() * TRAD_DAYS
    Sigma_annual = df.cov() * TRAD_DAYS
    assets = df.columns.tolist()
    n = len(assets)
    mu = mu_annual.loc[assets].values
    Sigma = Sigma_annual.loc[assets, assets].values
    return TRAD_DAYS, assets, n, mu, Sigma


def portfolio_stats(w: np.ndarray, rf=0.04237):
    """
    Parameters
    ----------
    w:                  np.ndarray, portfolio weights
    rf:                 float, risk-free rate, default current 10-year US yield
    Returns
    -------
    ret, vol, sharpe:   tuple, (return, volatility, Sharpe ratio)
    """
    ret = float(w @ mu)
    vol = float(np.sqrt(w @ Sigma @ w))
    sharpe = (ret - rf) / vol if vol > 0 else np.nan
    return ret, vol, sharpe


def max_div(w):
    """
    Parameters
    ----------
    w:                  np.ndarray, portfolio weights
    Returns
    -------
    Objective function  Diversification Ratio (DR) of the portfolio
    """
    vol = float(np.sqrt(w @ Sigma @ w)) #Portfolio volatility
    w_vol = float(w @ np.diag(Sigma)) #Sum of each asset's weight multiplied by its individual standard deviation
    return -vol/w_vol

In [5]:
#Implementation
try:
    df_ret = pd.read_csv("downloaded_data.csv")
    print("File found")
except:
    print("Start download")
    df_ret = data_download(TICKERS)

#Check data structure
print(df_ret.head())

TRAD_DAYS, assets, n, mu, Sigma = moments(df_ret)

#Boundaries and constraints
bounds = [(0.0, 1.0)] * n
cons = ({"type": "eq", "fun": lambda w: np.sum(w) - 1.0},)

#Maximum Diversification Portfolio (MDP)
#Initial values
w0 = np.repeat(1.0 / n, n)

mdp = minimize(max_div, w0, method="SLSQP", bounds=bounds, constraints=cons)
w_mdp = mdp.x
ret_mdp, vol_mdp, sharpe_mdp = portfolio_stats(w_mdp)

Start download
                 MMM       AXP      AMGN      AMZN      AAPL        BA  \
Date                                                                     
2025-11-14 -0.013690 -0.020918  0.002200 -0.012239 -0.001980 -0.000308   
2025-11-17 -0.013155 -0.045624  0.014651 -0.007785 -0.018338 -0.014030   
2025-11-18  0.002416 -0.001730  0.006650 -0.045329 -0.000075 -0.011430   
2025-11-19  0.004754  0.011615 -0.004633  0.000629  0.004179 -0.020942   
2025-11-20 -0.011897 -0.002440 -0.018660 -0.025238 -0.008639 -0.034626   

                 CAT       CVX      CSCO        KO  ...       NKE      NVDA  \
Date                                                ...                       
2025-11-14  0.000867  0.013027  0.007981  0.001266  ... -0.028573  0.017559   
2025-11-17 -0.003580 -0.017924 -0.002825 -0.006768  ... -0.019990 -0.018951   
2025-11-18 -0.009409  0.003325 -0.005285  0.007611  ... -0.006540 -0.028483   
2025-11-19  0.011327 -0.012577  0.013097 -0.001546  ...  0.002397  0.02

In [6]:
print("The Maximum Diversification Portfolio (MDP)")
print("===========================================")
print(f"Return: {ret_mdp:.3f}")
print(f"Volatility: {vol_mdp:.3f}")
print(f"Sharpe Ratio: {sharpe_mdp:.3f}")
print("===========================================")

The Maximum Diversification Portfolio (MDP)
Return: -0.039
Volatility: 0.279
Sharpe Ratio: -0.291
